# 301 — MCP Server and Tools

Demonstrates the Phase 3 MCP tool layer. Covers:

1. Direct Python calls to all exposed tools (shared, graph, risk, trace)
2. How to start the MCP server for Claude Desktop or other MCP clients
3. The full list of exposed MCP tools and their contracts

**Start the server** (outside this notebook) with:
```bash
python -m src.mcp.server
```

In [11]:
import sys
sys.path.insert(0, "..")

import json

from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tools.graph_tools import GraphTools
from src.tools.risk_tools import RiskTools
from src.tools.shared_tools import SharedTools
from src.tools.trace_tools import TraceTools

settings = get_neo4j_settings()
neo4j = Neo4jRepository(**vars(settings))
neo4j.test_connection()

graph   = GraphTools(neo4j)
risk    = RiskTools(neo4j)
trace   = TraceTools(TraceRepository(neo4j))
shared  = SharedTools(graph)

print("Tool layer ready")


def show(result):
    """Print a ToolResult summary and its structured data."""
    print(f"[{result.tool_name}] {'OK' if result.success else 'FAIL'} "
          f"({result.duration_ms} ms)")
    print(f"  {result.summary}")
    if result.data is not None:
        print(json.dumps(result.data, indent=2, default=str)[:1200])

Tool layer ready


## Shared tools

These orchestration-level tools support Phase 3 agent coordination.

In [12]:
# resolve_entity — canonicalise a company name
result = shared.resolve_entity("VODAFONE")
show(result)

if result.success and result.data.get("resolved"):
    COMPANY = result.data["canonical_name"]
    print(f"\nCanonical name for next cells: {COMPANY!r}")
else:
    COMPANY = "VODAFONE GROUP PLC"  # fallback
    print(f"\nUsing fallback name: {COMPANY!r}")

[resolve_entity] OK (20.2 ms)
  Resolved 'VODAFONE' → 'VODAFONE 2.' (#04083193, Active) [best fuzzy match, 10 candidate(s)].
{
  "resolved": true,
  "canonical_name": "VODAFONE 2.",
  "company_number": "04083193",
  "status": "Active",
  "exact_match": false,
  "match_count": 10,
  "all_matches": [
    {
      "name": "VODAFONE 2.",
      "company_number": "04083193",
      "status": "Active",
      "country_of_origin": "United Kingdom",
      "score": 6.454601764678955
    },
    {
      "name": "VODAFONE LIMITED",
      "company_number": "01471587",
      "status": "Active",
      "country_of_origin": "United Kingdom",
      "score": 6.454601764678955
    },
    {
      "name": "VODAFONE FOUNDATION",
      "company_number": "13199169",
      "status": "Active",
      "country_of_origin": "United Kingdom",
      "score": 6.454601764678955
    },
    {
      "name": "VODAFONE FINANCE LIMITED",
      "company_number": "03032761",
      "status": "Active",
      "country_of_origin": "Uni

In [13]:
# validate_plan — check steps before handing to an orchestrator
plan = [
    {"step_id": "step_1", "tool_name": "company_profile",           "parameters": {"company_name": COMPANY}},
    {"step_id": "step_2", "tool_name": "expand_ownership",          "parameters": {"company_name": COMPANY}},
    {"step_id": "step_3", "tool_name": "ownership_complexity_check", "parameters": {"company_name": COMPANY}},
    {"step_id": "step_4", "tool_name": "address_risk_check",         "parameters": {"company_name": COMPANY}},
    {"step_id": "step_5", "tool_name": "industry_context_check",     "parameters": {"company_name": COMPANY}},
    {"step_id": "step_6", "tool_name": "evaluate_stop_conditions",   "parameters": {}},
]
show(shared.validate_plan(plan))

print()

# Invalid plan — unknown tool name
bad_plan = [
    {"step_id": "step_1", "tool_name": "nonexistent_tool"},
    {"step_id": "",       "tool_name": "entity_lookup"},
]
show(shared.validate_plan(bad_plan))

[validate_plan] OK (0.0 ms)
  Plan is valid: 6 step(s) ready to execute.
{
  "valid": true,
  "step_count": 6,
  "valid_steps": [
    {
      "step_id": "step_1",
      "tool_name": "company_profile",
      "parameters": {
        "company_name": "VODAFONE 2."
      }
    },
    {
      "step_id": "step_2",
      "tool_name": "expand_ownership",
      "parameters": {
        "company_name": "VODAFONE 2."
      }
    },
    {
      "step_id": "step_3",
      "tool_name": "ownership_complexity_check",
      "parameters": {
        "company_name": "VODAFONE 2."
      }
    },
    {
      "step_id": "step_4",
      "tool_name": "address_risk_check",
      "parameters": {
        "company_name": "VODAFONE 2."
      }
    },
    {
      "step_id": "step_5",
      "tool_name": "industry_context_check",
      "parameters": {
        "company_name": "VODAFONE 2."
      }
    },
    {
      "step_id": "step_6",
      "tool_name": "evaluate_stop_conditions",
      "parameters": {}
    }
  ],
  "e

In [14]:
# evaluate_stop_conditions — decision gate after gathering signals
findings_complete = {
    "ownership_complexity": {"risk_level": "MEDIUM"},
    "control_signals":       {"risk_level": "HIGH"},
    "address_risk":          {"risk_level": "LOW"},
    "industry_context":      {"risk_level": "MEDIUM"},
}
show(shared.evaluate_stop_conditions(findings_complete))

print()

# Incomplete — two signals missing
findings_partial = {
    "ownership_complexity": {"risk_level": "LOW"},
    "address_risk":         {"risk_level": "LOW"},
}
show(shared.evaluate_stop_conditions(findings_partial))

[evaluate_stop_conditions] OK (0.0 ms)
  Investigation complete. All 4 signals gathered. Overall risk: HIGH — escalate for review.
{
  "should_stop": true,
  "escalate": true,
  "overall_risk": "HIGH",
  "signals_present": [
    "address_risk",
    "control_signals",
    "industry_context",
    "ownership_complexity"
  ],
  "signals_missing": [],
  "signal_count": 4,
  "required_count": 4
}

[evaluate_stop_conditions] OK (0.0 ms)
  Investigation incomplete: 2 signal(s) still needed: control_signals, industry_context.
{
  "should_stop": false,
  "escalate": false,
  "overall_risk": "LOW",
  "signals_present": [
    "address_risk",
    "ownership_complexity"
  ],
  "signals_missing": [
    "control_signals",
    "industry_context"
  ],
  "signal_count": 2,
  "required_count": 4
}


## Graph tools

In [15]:
# expand_ownership — walk the ownership graph
show(graph.expand_ownership(COMPANY, max_depth=3))

[expand_ownership] OK (42.9 ms)
  Found 6 ownership hop(s) across depths [1, 2, 3] for 'VODAFONE 2.'. All chains terminate at corporate entities — no individual UBOs found.
{
  "paths": [
    {
      "path_depth": 1,
      "hop": 1,
      "from_name": "VODAFONE INTERNATIONAL OPERATIONS LIMITED",
      "from_labels": [
        "Company"
      ],
      "ownership_pct_min": 75,
      "ownership_pct_max": 100,
      "ownership_controls": [
        "ownership-of-shares-75-to-100-percent"
      ],
      "to_name": "VODAFONE 2.",
      "to_labels": [
        "Company"
      ]
    },
    {
      "path_depth": 2,
      "hop": 1,
      "from_name": "VODAFONE EUROPEAN INVESTMENTS",
      "from_labels": [
        "Company"
      ],
      "ownership_pct_min": 75,
      "ownership_pct_max": 100,
      "ownership_controls": [
        "ownership-of-shares-75-to-100-percent"
      ],
      "to_name": "VODAFONE INTERNATIONAL OPERATIONS LIMITED",
      "to_labels": [
        "Company"
      ]
    },
    

In [16]:
show(graph.company_profile(COMPANY))

[company_profile] OK (69.4 ms)
  VODAFONE 2. (#04083193, Active). SIC codes: 96090 (Other service activities n.e.c.). Direct owners: VODAFONE INTERNATIONAL OPERATIONS LIMITED.
{
  "company": {
    "name": "VODAFONE 2.",
    "company_number": "04083193",
    "status": "Active",
    "country_of_origin": "United Kingdom"
  },
  "address": {
    "address_line_1": "VODAFONE HOUSE",
    "address_line_2": "THE CONNECTION",
    "post_town": "NEWBURY",
    "county": "BERKSHIRE",
    "post_code": "RG14 2FN",
    "country": null
  },
  "sic_codes": [
    {
      "sic_code": "96090",
      "sic_description": "Other service activities n.e.c."
    }
  ],
  "direct_owners": [
    {
      "owner_name": "VODAFONE INTERNATIONAL OPERATIONS LIMITED",
      "owner_labels": [
        "Company"
      ],
      "ownership_pct_min": 75,
      "ownership_pct_max": 100,
      "ownership_controls": [
        "ownership-of-shares-75-to-100-percent"
      ]
    }
  ]
}


## Risk tools

In [17]:
for fn in [
    lambda: risk.ownership_complexity_check(COMPANY),
    lambda: risk.control_signal_check(COMPANY),
    lambda: risk.address_risk_check(COMPANY),
    lambda: risk.industry_context_check(COMPANY),
]:
    r = fn()
    print(f"[{r.tool_name}] risk_level={r.data.get('risk_level') if r.data else 'n/a'}  —  {r.summary}")

[ownership_complexity_check] risk_level=HIGH  —  Ownership chain for 'VODAFONE 2.': max depth 3, 3 unique owner(s), 0 individual UBO(s). Complexity risk: HIGH. All chains terminate at corporate entities — no individual UBOs resolved.
[control_signal_check] risk_level=LOW  —  'VODAFONE 2.' ownership is via standard share ownership only. Control risk: LOW.
[address_risk_check] risk_level=MEDIUM  —  'VODAFONE 2.' shares address (RG14 2FN) with 48 other companies (46 active, 2 dissolved — 4% dissolution rate). Address risk: MEDIUM.
[industry_context_check] risk_level=LOW  —  'VODAFONE 2.' SIC codes ['96090'] are standard. Peer dissolution rate: 17% (34/200). Industry risk: LOW.


## Trace tools

Retrieval path: `list_recent_traces` → pick a `trace_id` → `retrieve_trace`.

In [18]:
recent = trace.list_recent_traces(limit=5)
show(recent)

trace_id = None
if recent.success and recent.data:
    trace_id = recent.data[0]["trace_id"]
    print(f"\nLatest trace_id: {trace_id}")

[list_recent_traces] OK (13.8 ms)
  No traces stored yet.
[]


In [19]:
if trace_id:
    show(trace.retrieve_trace(trace_id))
else:
    print("No traces stored yet — run notebooks 207–210 first to generate trace data.")

No traces stored yet — run notebooks 207–210 first to generate trace data.


In [20]:
# find_traces_by_entity — entity-centric trace lookup
show(trace.find_traces_by_entity(COMPANY))

[find_traces_by_entity] OK (16.6 ms)
  No traces found connected to 'VODAFONE 2.'.
[]


## MCP server

### Starting the server

```bash
# stdio transport (for Claude Desktop / MCP-compatible clients)
python -m src.mcp.server

# or via the MCP CLI
mcp run src/mcp/server.py

# inspect the tool manifest
mcp dev src/mcp/server.py
```

### Claude Desktop integration

Add to `~/.claude/claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "entity-risk-ai": {
      "command": "python",
      "args": ["-m", "src.mcp.server"],
      "cwd": "/path/to/entity-risk-ai",
      "env": {
        "NEO4J_URI": "bolt://localhost:7687",
        "NEO4J_USERNAME": "neo4j",
        "NEO4J_PASSWORD": "your_password",
        "NEO4J_DATABASE": "neo4j"
      }
    }
  }
}
```

In [24]:
# Inspect the registered MCP tools without starting the server
import sys
sys.path.insert(0, "..")

# Import the server module to access the registered tools
from src.mcp.server import mcp

print(f"Server name : {mcp.name}")
print()

# List all registered tools via the internal tool registry
tools = mcp._tool_manager._tools
print(f"Exposed MCP tools ({len(tools)}):")
print()
for name, tool in sorted(tools.items()):
    desc_first_line = (tool.description or "").splitlines()[0]
    print(f"  {name:<35}  {desc_first_line}")

Server name : entity-risk-ai

Exposed MCP tools (15):

  address_risk_check                   
  company_profile                      
  control_signal_check                 
  entity_lookup                        
  evaluate_stop_conditions             
  expand_ownership                     
  find_traces_by_entity                
  industry_context_check               
  list_recent_traces                   
  ownership_complexity_check           
  resolve_entity                       
  retrieve_trace                       
  shared_address_check                 
  sic_context                          
  validate_plan                        


In [25]:
neo4j.close()
print("Driver closed")

Driver closed
